# Ollama Local LLM — Quick Test

Tests: server connection, model config, generate, chat, streaming, embeddings, and performance.

In [ ]:
## Cell 1: Config — change MODEL_NAME to switch models
import requests, json, time

OLLAMA_BASE_URL = "http://localhost:11434"
MODEL_NAME = "qwen2.5-coder:7b"          # <<< Change this to use a different model

DEFAULT_OPTIONS = {
    "temperature": 0.7,
    "top_p": 0.9,
    "top_k": 40,
    "num_predict": 100,
    "repeat_penalty": 1.1,
}
print(f"Model: {MODEL_NAME}  |  Server: {OLLAMA_BASE_URL}")

In [ ]:
## Cell 2: Server connection + list models
r = requests.get(f"{OLLAMA_BASE_URL}/", timeout=5)
assert r.status_code == 200, "Ollama not running!"
print(f"✅ Server: {r.text.strip()}")

models = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=10).json()["models"]
model_names = [m["name"] for m in models]
for m in models:
    tag = " ◀ SELECTED" if m["name"] == MODEL_NAME else ""
    print(f"  • {m['name']:30s} ({m['size']/1e9:.1f} GB){tag}")

assert MODEL_NAME in model_names, f"'{MODEL_NAME}' not found! Run: ollama pull {MODEL_NAME}"

In [ ]:
## Cell 3: Generate (non-streaming) with config options
data = requests.post(f"{OLLAMA_BASE_URL}/api/generate", json={
    "model": MODEL_NAME,
    "prompt": "What is 2+2? Answer briefly.",
    "stream": False,
    "options": DEFAULT_OPTIONS,
}, timeout=120).json()

print(f"Response: {data['response']}")
print(f"Tokens: {data.get('eval_count',0)} | Speed: {data.get('eval_count',0)/(data.get('eval_duration',1))*1e9:.1f} tok/s")

In [ ]:
## Cell 4: Generate (streaming)
r = requests.post(f"{OLLAMA_BASE_URL}/api/generate", json={
    "model": MODEL_NAME, "prompt": "Count 1 to 5.", "stream": True,
    "options": {"num_predict": 30, "temperature": 0},
}, stream=True, timeout=120)
tokens = []
for line in r.iter_lines(decode_unicode=True):
    if line:
        c = json.loads(line)
        tokens.append(c.get("response", ""))
        if c.get("done"): break
print(f"Streamed {len(tokens)} chunks → {''.join(tokens)}")
print("✅ Streaming works!" if len(tokens) > 1 else "⚠️ Only 1 chunk")

In [ ]:
## Cell 5: Chat — single-turn + multi-turn
# Single turn
r1 = requests.post(f"{OLLAMA_BASE_URL}/api/chat", json={
    "model": MODEL_NAME, "stream": False,
    "messages": [{"role": "user", "content": "Hello!"}],
    "options": {"num_predict": 30},
}, timeout=120).json()
print(f"Single-turn: {r1['message']['content']}")

# Multi-turn
msgs = [
    {"role": "system", "content": "You are a math tutor."},
    {"role": "user", "content": "What is 3*7?"},
]
r2 = requests.post(f"{OLLAMA_BASE_URL}/api/chat", json={
    "model": MODEL_NAME, "messages": msgs, "stream": False,
    "options": {"num_predict": 30, "temperature": 0},
}, timeout=120).json()
print(f"Turn 1: {r2['message']['content']}")

msgs += [r2["message"], {"role": "user", "content": "Multiply that by 2."}]
r3 = requests.post(f"{OLLAMA_BASE_URL}/api/chat", json={
    "model": MODEL_NAME, "messages": msgs, "stream": False,
    "options": {"num_predict": 30, "temperature": 0},
}, timeout=120).json()
print(f"Turn 2: {r3['message']['content']}")

In [ ]:
## Cell 6: System prompt test
data = requests.post(f"{OLLAMA_BASE_URL}/api/generate", json={
    "model": MODEL_NAME,
    "prompt": "Who are you?",
    "system": "You are a pirate. Always answer in pirate speak.",
    "stream": False,
    "options": {"num_predict": 50, "temperature": 0.7},
}, timeout=120).json()
print(f"Pirate response: {data['response']}")

In [ ]:
## Cell 7: Embeddings
embed_model = next((n for n in model_names if "embed" in n.lower()), MODEL_NAME)
data = requests.post(f"{OLLAMA_BASE_URL}/api/embed", json={
    "model": embed_model, "input": ["Hello", "World", "Test"],
}, timeout=60).json()
embs = data["embeddings"]
print(f"Model: {embed_model} | {len(embs)} texts → dim {len(embs[0])}")
print(f"Sample: {embs[0][:5]}")

In [ ]:
## Cell 8: Performance benchmark
t0 = time.time()
data = requests.post(f"{OLLAMA_BASE_URL}/api/generate", json={
    "model": MODEL_NAME, "prompt": "Explain gravity in 2 sentences.",
    "stream": False, "options": {"num_predict": 60},
}, timeout=120).json()
wall = time.time() - t0
toks = data.get("eval_count", 0)
tok_s = toks / data.get("eval_duration", 1) * 1e9
prompt_ms = data.get("prompt_eval_duration", 0) / 1e6

print(f"Wall time    : {wall:.2f}s")
print(f"Prompt eval  : {prompt_ms:.1f}ms")
print(f"Tokens       : {toks}")
print(f"Throughput   : {tok_s:.1f} tok/s")
print(f"\n✅ All tests passed for {MODEL_NAME}!")